# Pro-bono: Student Reading Campaign Proposal

Author: Engineering Lead (Pro-bono Team)
Date: 2025-09-28

**Executive summary**

This notebook proposes a practical system to increase student reading by combining librarian expertise, an automated recommendation engine (semantic + TF-IDF fallbacks), and campaign-driven nudges surfaced via a simple Gradio UI. It sketches architecture, ingestion, vectorization into Qdrant (or seed file), API endpoints, a Gradio campaign UI (port 7863), monitoring, and rollout estimates. It contains runnable prototypes (guarded) and diagrams suitable for partner review.

## 1) Use case & stakeholders

- Stakeholders: Librarians, Students, School IT, District Admins, Partner NGOs.
- Pain points: Low visibility into what books resonate, limited time for librarians to curate at scale, difficulty measuring impact across schools.
- Goals: Increase student reads by surfacing librarian-approved recommendations, measure campaign impact, and maintain privacy and low-cost deployment.

## Scaling the POC to AWS or GCP

Below is a practical checklist and service mapping for taking this lightweight POC to a production-grade deployment on a major cloud provider (AWS or GCP). The recommendations emphasize low-cost, secure, and observable architecture while supporting model serving, vector search, and audit event persistence.

### High-level architecture
- Frontend: Gradio apps (student & campaign UIs) — run in containers behind a managed ingress/load balancer (ALB/GCLB). Consider converting to a small React app + hosted static site for scale and lower cost; call the API for recommendations.
- API: FastAPI recommendation + campaign endpoints — containerized, autoscaled behind policy-based autoscaling (CPU/RPS). Use managed container services (AWS ECS/Fargate or GCP Cloud Run) for quick ops and lower maintenance.
- Model serving / LLM judge: options depend on model size and latency needs:
  - Hosted LLMs (low ops): OpenAI/Anthropic/Vertex AI/Bedrock — easiest to operate, predictable SLAs, but may have cost/privacy tradeoffs.
  - Managed on-cloud inference (medium ops): AWS Sagemaker Endpoints / GCP Vertex AI Model Serving — supports custom containers and autoscaling, good for larger models and fine-tuning workflows.
  - Self-hosted on GPU instances (higher ops): EC2/GCE with GPUs (e.g., g5, p4, A100) or inference-optimized instances; use Triton or a lightweight HTTP wrapper (Hermes3/llama-server) for the judge. Consider EC2 Spot + autoscaling for cost savings.
- Vector store: production-ready options include Qdrant (self-hosted in containers/VMs) or managed vector DBs such as Pinecone, Weaviate Cloud, or Qdrant Cloud. On AWS/GCP you can also run a HA Qdrant cluster in Kubernetes (EKS/GKE) with persistent volumes.
- Persistent store & auditing: PostgreSQL (RDS / Cloud SQL) for audit_logs/judge_logs, or continue with SQLite for low-volume pilots. Use S3/GCS for artifact storage (catalog exports, model files).
- Observability: Prometheus + Grafana for metrics, Loki or cloud logging for logs, and alerting on error rates, model latency, and judge failures. Use CloudWatch / Stackdriver integrations when using managed services.

### Service mapping (quick)
- AWS: ECS/Fargate or EKS for containers, RDS (Postgres) for DB, S3 for object storage, SageMaker or EC2 GPU for model serving, Elastic Load Balancer / API Gateway for ingress, CloudWatch + X-Ray for telemetry.
- GCP: Cloud Run or GKE for containers, Cloud SQL (Postgres) for DB, Cloud Storage for artifacts, Vertex AI for hosted model serving or GCE GPU instances for self-hosting, Cloud Monitoring / Logging for telemetry.

### Security & compliance
- Use IAM and service accounts with least-privilege. Put the judge/model-serving in a private subnet and only expose a stable, authenticated inference API to the application layer.
- Encrypt data at rest (RDS, S3/GCS) and in transit (TLS everywhere).
- Consider VPC peering / Private Service Connect to keep internal traffic off the public internet when integrating managed model providers or vector DBs.
- For student data, treat PII carefully: anonymize student_id in exports, audit access via role-based controls, and consider a Data Protection Addendum for third-party LLM providers.

### Cost & performance tradeoffs
- Hosted LLMs: pay-per-request, simple ops, but recurring cost may be high for heavy use. Good for early pilots.
- Self-hosted GPU inference: higher fixed infra cost but cheaper per-inference at scale; requires MLOps (model upgrades, monitoring).
- Vector DB choices: managed stores reduce ops burden. Self-hosted Qdrant on k8s is flexible but requires ops resources.

### Rollout checklist (minimal viable to production)
1. Replace SQLite with managed Postgres (RDS/Cloud SQL) and migrate `audit_logs` + `judge_logs`.
2. Containerize Gradio + API (Docker) and deploy to Cloud Run / ECS Fargate with HTTPS endpoint and basic auth or OIDC auth for admin UIs.
3. Choose model-serving approach: start with a hosted LLM provider for judge scoring, then pilot an on-cloud managed model (Vertex AI / SageMaker) if cost/privacy needs change.
4. Move seed catalog and artifacts to S3/GCS; configure CI to upsert vectors to managed vector DB or self-hosted Qdrant cluster.
5. Add monitoring + alerting (errors, judge-empty-responses, high latency, queue backlogs) and log retention policies.
6. Harden access controls (RBAC for Admin UI), enable audit logging, and validate PII handling and data retention policies.
7. Run a 2-week pilot with real librarian workflows, collect metrics (CTR, borrow_rate uplift), and iterate on judge prompts and autos-approval thresholds.

### Example infra cost considerations (very approximate)
- Small pilot (low traffic): $100–$1,000/month using Cloud Run/Fargate + managed DB + hosted LLM usage (light).
- Moderate scale (district): $1k–$10k/month depending on LLM usage; self-hosted inference with spot instances can reduce model costs.
- Large scale (many schools): costs dominated by model serving and vector DB throughput; expect $10k+/month unless using heavy on-prem GPU infra.

### Final notes
- Start with the lowest ops path that meets privacy requirements (hosted LLMs + managed DB + Cloud Run). If costs grow, consider migrating judge/model-serving to managed on-cloud or self-hosted GPU instances with autoscaling and cost controls.
- I can provide a concrete Terraform / CloudFormation starter template and a migration checklist if you want to move forward with either AWS or GCP.
